# Tutorial PXCT data analysis (PART 2)

### Tutor: Julio C. da Silva (Néel Institute CNRS, Grenoble, France) 
### email: julio-cesar.da-silva@neel.cnrs.fr
#### Personal webpage: https://sites.google.com/view/jcesardasilva

### <span style="color:red">** Disclaimer: This notebook is intended from educational reasons only.**</span>
<span style="color:red">**Warning: You should have completed part 1 before starting part 2**</span>

<table class="tfo-notebook-buttons" align="center">
  <td>
    <a target="_blank" rel="noopener noreferrer" href="https://github.com/jcesardasilva/toupy"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
</table>

#### Importing packages again
Since we start a new notebook, we need to import the packages again:

In [ ]:
# standard packages
import time
# third party packages
from IPython import display as ipy_display
import matplotlib.pyplot as plt
import numpy as np
import toupy

### Interactive backend

Activate the interactive matplotlib backend (required for GUI tools):

In [ ]:
%matplotlib widget

#### Let us reload our data 
We do this the same way we did in Part 1, but we only change the filename to `PXCTcorrprojections.npz`:

In [ ]:
fname = 'PXCTcorrprojections.npz'
data_dict = np.load(fname) # load the file
list(data_dict.files) # this one list the keys of the data dictionary extracted from the file
wavelen = data_dict['wavelen']
pixsize = data_dict['psize']
theta = data_dict['theta']
projections = data_dict['projections'].astype(np.float32) # <- ATTENTION: this one is memory consuming. 
nproj, nr, nc = projections.shape
delta_theta = np.diff(np.sort(theta))[0]

print(f"The total number of projections is {nproj}")
print(f"The angular sampling interval is {delta_theta:.02f} degrees")
print(f"The projection pixel size of the projections is {pixsize/1e-9:.02f} nm")
print(f"The wavelenth of the incoming photons is {wavelen/1e-10:.02f} Angstroms")

In [ ]:
plt.close('all')
fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
im1 = ax1.imshow(projections[0], cmap='bone', vmin=-4, vmax=1)
ax1.set_title('Phase proj. at 0°', fontsize=13)
ax1.axis('off')
im2 = ax2.imshow(projections[-1], cmap='bone', vmin=-4, vmax=1)
ax2.set_title(f'Phase proj. at {theta[-1]:.1f}°', fontsize=13)
ax2.axis('off')
fig1.colorbar(im1, ax=[ax1, ax2], shrink=0.8, label='Phase shift (rad)')
plt.show()

### Step 3 — Alignment of the tomographic projections

Even with the best experimental setup, small mechanical vibrations and drifts cause the sample to shift slightly between projections. These **misalignments** produce blurring, streak artefacts, and reduced resolution in the 3-D reconstruction.

We correct the alignment in two independent steps:
1. **Vertical alignment** — uses the Helgason–Ludwig consistency condition.
2. **Horizontal alignment** — uses the tomographic consistency condition, which exploits the uniqueness of the sinogram-reconstruction relationship.

All alignment shifts are stored in a single array `shiftproj` of shape `(2, nproj)`:
- `shiftproj[0, :]` — vertical shifts (in pixels)
- `shiftproj[1, :]` — horizontal shifts (in pixels)

### Alignment of the tomographic projections
Great! Now that we have the projections corrected by the linear phase ramp and all unwrapped, we can start the alignment of the projections (registration in the language of digital signal processing). 

We will do it in the vertical and horizontal direction "independently", starting with the vertical alignment. 

#### Vertical alignment — Helgason–Ludwig consistency condition

The **Helgason–Ludwig (HL) condition** states that the integral of any parallel-beam projection along the radial direction is independent of the projection angle θ. In other words:

$$\int_{-\infty}^{\infty} p(t, \theta)\, dt = \text{const} \quad \forall\, \theta$$

where p(t, θ) is the projection at angle θ. If the sample drifts vertically between projections, this integral changes — we can detect and correct those drifts by minimising the variance of the projection integrals across angles.

**In practice**: `alignprojections_vertical` iteratively shifts each projection so that all row-integrals converge to a common value.

The vertical alignment is performed based on the **Helgason-Ludwig consistency condition**. This is basically the Plancherel’s theorem for the Radon transform and states that the integral of any projection along horizontal directions is independent of the angle θ. 

In [ ]:
# import the toupy routines we will need
from toupy.registration import alignprojections_vertical

This step requires a certain number of parameters. For this reason, let us create a dictionary of parameters:

In [ ]:
params = dict() # initializing dictionary
params["pixtol"] = 0.01  # Tolerance of registration in pixels (E.g. 0.1 means 1/10 of pixel)
params["polyorder"] = 2  # Polynomial order to remove bias (E.g. 2 means 2nd order polynomial)
params["shiftmeth"] = "linear" # "linear" = bilinear interpolation or "fourier" = shift in the Fourier space
params["maxit"] = 10  # max of iterations
params["deltax"] = 40  # From edge of region to edge of image in x
params["limsy"] = (70, 200) # Vertical extent of the region to be considered in the alignment

Now that we have the parameters set, let us start the alignment. For we need to create an array which will keep the shifts to be applied to the projections in order to align them. This array, `shiftproj` which have a shape as `(2,nproj)`. Thus, `shiftproj[0]` will contain the shifts for the vertical diretions whereas `shiftproj[1]` will contains the shifts for the horizontal direction.

In [ ]:
# initializing shiftstack with zeros
shiftproj = np.zeros((2,nproj))
print(f"The shape of shifproj is {shiftproj.shape}")

In [ ]:
shiftproj, valignproj = alignprojections_vertical(projections,shiftproj,**params)

In [ ]:
from IPython import display as ipy_display

# Show a short animation of projections before and after vertical alignment
disp_range = range(300, 330)

fig3, (ax31, ax32) = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
im31 = ax31.imshow(projections[0], cmap='bone', vmin=-4, vmax=1)
ax31.set_title('Original — proj. 1', fontsize=14)
ax31.axis('off')
im32 = ax32.imshow(valignproj[0], cmap='bone', vmin=-4, vmax=1)
ax32.set_title('Vert. aligned — proj. 1', fontsize=14)
ax32.axis('off')

import time
for ii in disp_range:
    im31.set_data(projections[ii])
    ax31.set_title(f'Original — proj. {ii}', fontsize=14)
    im32.set_data(valignproj[ii])
    ax32.set_title(f'Vert. aligned — proj. {ii}', fontsize=14)
    ipy_display.display(fig3)
    ipy_display.clear_output(wait=True)
    time.sleep(0.05)

In [ ]:
from IPython import display
#-------
# you can put here the projections you want to display
disp = [300,329] # starts at index 0.
#-------
# preparing figure canvas
plt.close('all')
fig3 = plt.figure(3, figsize = (10,4), constrained_layout=True)
ax31 = fig3.add_subplot(121)
im31 = ax31.imshow(projections[0],cmap='bone', vmin = -4, vmax = 1)
ax31.set_title('Projection number 1', fontsize = 16)
ax32 = fig3.add_subplot(122)
im32 = ax32.imshow(valignproj[0],cmap='bone', vmin = -4, vmax = 1)
ax32.set_title('Vert. aligned projec. number 1', fontsize = 16)
for ii in range(disp[0],disp[-1]+1):
    im31.set_data(projections[ii])
    ax31.set_title(f'Projection number {ii}', fontsize = 16)
    im32.set_data(valignproj[ii])
    ax32.set_title(f'Vert. aligned projec. number {ii}', fontsize = 16)
    display.display(plt.gcf())
    display.clear_output(wait=True)
    time.sleep(0.001)

In [ ]:
from IPython import display as ipy_display

sinogram = valignproj[:, 300, :]
argangle = np.argsort(theta)
sinosort = sinogram[argangle]

fig4, (ax41, ax42) = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
ax41.imshow(sinogram.T, cmap='bone', vmin=-4, vmax=1, aspect='auto')
ax41.set_xlabel('Projection number'); ax41.set_ylabel('Radial coordinate')
ax41.set_title('Original sinogram (acquisition order)', fontsize=14)
ax42.imshow(sinosort.T, cmap='bone', vmin=-4, vmax=1, aspect='auto')
ax42.set_xlabel('Projection number'); ax42.set_ylabel('Radial coordinate')
ax42.set_title('Sinogram sorted by angle', fontsize=14)
plt.show()

The **sorted sinogram** should show smooth, continuous sinusoidal curves. Discontinuities or jumps indicate remaining alignment errors. The sorted sinogram is also what is passed to the FBP reconstruction algorithm.

### Preparation for horizontal alignment

For horizontal alignment we use the **tomographic consistency condition**: if we reconstruct a slice from the sinogram and then re-project it, the resulting sinogram must equal the input sinogram. Any difference reveals horizontal misalignment.

To make this condition robust against phase wraps and noise, we work with the **derivative of the projections** (equivalent to passing a high-pass filter that suppresses slowly varying artefacts).

The horizontal alignment has three sub-steps:
1. Compute derivatives of the vertically aligned projections.
2. Find a first estimate of the rotation axis offset by reconstructing one slice.
3. Iteratively refine the horizontal shifts using tomographic consistency on multiple slices.

In [ ]:
plt.imshow(sinosort.T[50:-50],cmap='bone', vmin = -4, vmax = 1)
plt.xlabel('Projection number')
plt.ylabel('Radial coordinate')
plt.axis('tight')

### Preparation for the horizontal alignment
For the horizontal alignment, we will use the tomographic consistency condition which reflects the uniqueness between the sinogram and the tomographic reconstructed slice, i.e., for each reconstructed slice, there is only one sinogram that can correspond to that reconstruction. 

Therefore, after reconstructing a slice and re-projecting to obtain the sinogram from the reconstructed slice, the resulting sinogram must be equal to the initial sinogram used for the reconstruction. If this is not the case, this will mean that the projections are horizontally misaligned.

Consequently, we need to be able to reconstruct a tomographic slice from the data we have so far. We will not do it using a non-standard tomographic reconstruction approach which is insensitive to spikes or possible phase wraps not yet corrected. For this, we will use the derivatives of the projections. 

#### Calculating the derivative of the projections

In [ ]:
# import the toupy routines we will need
from toupy.restoration import calculate_derivatives, chooseregiontoderivatives

Entering the required parameters

In [ ]:
params = dict() # initializing dictionary
params["deltax"] = 55  # From edge of region to edge of image in x
params["limsy"] = (55, 450)  # (top, bottom)
params["shift_method"] = "fourier" # "linear" = bilinear interpolation or "fourier" = shift in the Fourier space

In [ ]:
roix, roiy = chooseregiontoderivatives(valignproj, **params)

In [ ]:
from IPython import display as ipy_display

disp_range = range(300, 320)
fig5, (ax51, ax52) = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
im51 = ax51.imshow(valignproj[0, roiy[0]:roiy[-1], roix[0]:roix[-1]], cmap='bone')
ax51.set_title('Proj. 1 (ROI)', fontsize=14)
ax51.axis('off')
im52 = ax52.imshow(valigndiff[0], cmap='bone', vmin=-0.15, vmax=0.15)
ax52.set_title('Derivative of proj. 1', fontsize=14)
ax52.axis('off')

import time
for ii in disp_range:
    im51.set_data(valignproj[ii, roiy[0]:roiy[-1], roix[0]:roix[-1]])
    ax51.set_title(f'Proj. {ii} (ROI)', fontsize=14)
    im52.set_data(valigndiff[ii])
    ax52.set_title(f'Derivative of proj. {ii}', fontsize=14)
    ipy_display.display(fig5)
    ipy_display.clear_output(wait=True)
    time.sleep(0.05)

#### Estimating the rotation axis position

The **rotation axis** of the tomograph may not be perfectly centred on the detector. An offset of even 1–2 pixels causes a characteristic "double-edge" blurring in the reconstruction.

`estimate_rot_axis` reconstructs one slice using FBP with a trial offset, then displays it alongside the sinogram so you can visually identify the offset that gives the sharpest, artefact-free reconstruction.

**What to look for**: at the correct offset, the reconstructed particle boundary is sharp and the background is uniform. A wrong offset gives a doubled or smeared boundary.

In [ ]:
#------------
# parameters
#------------
# you can put here the projections you want to display
disp = [200,320] # starts at index 0.
#------------
# preparing figure canvas
plt.close('all')
fig3 = plt.figure(3, figsize = (10,4), constrained_layout=True)
ax31 = fig3.add_subplot(121)
im31 = ax31.imshow(valignproj[0,roiy[0]:roiy[-1],roix[0]:roix[-1]],cmap='bone')
ax31.set_title('Projection number 1', fontsize = 16)
ax32 = fig3.add_subplot(122)
im32 = ax32.imshow(valigndiff[0],cmap='bone',vmin=-0.15, vmax=0.15)
ax32.set_title('Derivative of projec. number 1', fontsize = 16)
for ii in range(disp[0],disp[-1]+1):
    im31.set_data(valignproj[ii,roiy[0]:roiy[-1],roix[0]:roix[-1]])
    ax31.set_title(f'Projection number {ii}', fontsize = 16)
    im32.set_data(valigndiff[ii])
    ax32.set_title(f'Derivative of projec. number {ii}', fontsize = 16)
    display.display(plt.gcf())
    display.clear_output(wait=True)
    time.sleep(0.001)

#### Checking for the rotation axis positions
We will now visually check the rotation axis and get a preliminary approximation of its position. For this, we will now reconstruct our first tomographic slice, yet likely misaligned. For this reconstruction, we will use a modified Filtered Back Projection (FBP) algorithm, which accepts the derivatives of the projections.

The tomographic reconstruction will be performed on a CPU for this tutorial. But it can be accelerated much more if implemented on GPU cards. 

In [ ]:
# import the toupy routines we will need
from toupy.registration import estimate_rot_axis

In [ ]:
params["slicenum"] = 225  # Choose the slice
params["filtertype"] = "hann"  # Filter to use for FBP
params["freqcutoff"] = 0.9  # Normalized frequency cutoff in case you want to apply a low pass band filter
params["circle"] = True # Apply a circular region in the external part of the slice
params["algorithm"] = "FBP" # Filtered Back projections algorithm
# initial guess of the offset of the axis of rotation
params["rot_axis_offset"] = 0
params["cliplow"] = None  # clip on low threshold for display
params["cliphigh"] = -1e-4  # clip on high threshold for display
params["sinohigh"] = None  # -0.1 # maximum gray level to display the sinograms
params["sinolow"] = None  # 0.1 # minimum gray level to display the sinograms
params["sinocmap"] = "bone" # sinogram colormap
params["colormap"] = "bone" # slice colormap
params["derivatives"] = True # True if the input is the derivative of the projections
params["calc_derivatives"] = False  # Calculate derivatives if not done

In [ ]:
estimate_rot_axis(valigndiff, theta, **params)

In [ ]:
# We should then set the parameter:
params["rot_axis_offset"] = 18

In [ ]:
from IPython import display as ipy_display

shiftproj[1] = np.zeros(valigndiff.shape[0]) + params["rot_axis_offset"]
fig6, (ax61, ax62) = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
ax61.plot(shiftproj_orig.T)
ax61.legend(['Vertical', 'Horizontal'])
ax61.set_xlabel('Projection number'); ax61.set_ylabel('Shift (pixels)')
ax61.set_title('Shifts before rotation-axis correction', fontsize=13)
ax61.grid(True, alpha=0.3)
ax62.plot(shiftproj.T)
ax62.legend(['Vertical', 'Horizontal'])
ax62.set_xlabel('Projection number'); ax62.set_ylabel('Shift (pixels)')
ax62.set_title(f'Shifts with rot. axis offset = {params["rot_axis_offset"]} px', fontsize=13)
ax62.grid(True, alpha=0.3)
plt.show()

In [ ]:
shiftproj[1] = np.zeros(valigndiff.shape[0]) # to be used in case of accidental overwritting
shiftproj_orig = shiftproj.copy() # keeping track of the original shiftproj

In [ ]:
shiftproj[1] = np.zeros(valigndiff.shape[0]) + params["rot_axis_offset"] # note we have reinforced the zero initialization to prevent accidents
fig4 = plt.figure(4,(10,4), constrained_layout=True)
ax41 = fig4.add_subplot(121)
ax41.plot(shiftproj_orig.T)
ax41.legend(['vertical', 'horizontal'])
ax41.set_title('Before the offset of the rotation axis')
ax42 = fig4.add_subplot(122)
ax42.plot(shiftproj.T)
ax42.legend(['vertical', 'horizontal'])
ax42.set_title('Before the offset of the rotation axis')
display.display(plt.gcf())
display.clear_output(wait=True)

Great! Now we can begin the horizontal alignment procedure:

### Actual horizontal alignement

Now, we will start the **horizontal alignment**.

In [ ]:
# import the toupy routines we will need
from toupy.registration import (
    alignprojections_horizontal,
    compute_aligned_horizontal,
    oneslicefordisplay,
    refine_horizontalalignment,
    tomoconsistency_multiple,
)

I pasted some of the parameters below in order we can easily changed them if needed. We will already enter our prelimary estimate of the rotation axis position:

In [ ]:
params["slicenum"] = 225  # Choose the slice
params["filtertype"] = "hann"  # Filter to use for FBP
#params["freqcutoff"] = 0.2#0.4  # Frequency cutoff (between 0 and 1)
params["freqcutoff_schedule"] = [0.3, 0.5, 0.7] # Multi-stage frequency-cutoff schedule 
params["circle"] = True # Apply a circular region in the external part of the slice
params["rot_axis_offset"] = 18 #<---- our estimate goes here --------
params["pixtol"] = 0.01  # Tolerance of registration in pixels (not resolution)
params["shiftmeth"] = "fourier"  # 'sinc' or 'linear' better for noise
params["maxit"] = 100  # max of iterations
params["cliplow"] = None  # clip air threshold
params["cliphigh"] = -4e-4  # clip on sample threshold
params["sinohigh"] = None
params["sinolow"] = None
params["multiresolution"] = True # Enable the coarse-to-fine warm-start
params["mr_factor"] = 2        # Downsampling factor for the coarse sinogram
params["n_coarse_iter"] = 50    # Max iterations at coarse resolution

In [ ]:
# calculate the sinogram needed for the alignment
sinogram = np.transpose(valigndiff[:, params["slicenum"], :]).copy()
shiftproj = alignprojections_horizontal(sinogram, theta, shiftproj, **params)

In [ ]:
# alignment refinement with different parameters if necessary
params["freqcutoff"] = 0.7  # Frequency cutoff (between 0 and 1)
params["rtol"] = 1e-3
shiftstack, params = refine_horizontalalignment(
        valigndiff, theta, shiftproj, **params
)

In [ ]:
import os
os.cpu_count()

Let us now repeat this alignment for 10 slices (-5 and +5 relative to the currently selected slicenum). At the end, we can decide to use (or not) the average of the shift values. 

In [ ]:
params["n_slices_tc"] = 20 # 20 slices centered on slicenum (default: 10)
params["n_workers_tc"] = 4   # 4 processes in parallel (default: cpu_count // 2)
# params["n_workers_tc"] = 1  # sequential, for debugging
params["silent"] = True
shiftproj = tomoconsistency_multiple(valigndiff, theta, shiftproj, **params)

Very good, we have reached a good alignment. Therefore, we now apply the shifts to the projections and reconstruct one tomographic slice for our inspection:

In [ ]:
alignedproj = compute_aligned_horizontal(
        valigndiff, shiftproj, shift_method=params["shiftmeth"]
    )

In [ ]:
from IPython import display as ipy_display

sinoalig = originalproj[:, params['slicenum'], :]
sinoaligsort = sinoalig[np.argsort(theta)]

fig7, (ax71, ax72) = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
ax71.imshow(sinosort.T[50:-50], cmap='bone', vmin=-4, vmax=1, aspect='auto')
ax71.set_xlabel('Projection number'); ax71.set_ylabel('Radial coordinate')
ax71.set_title('Before horizontal alignment', fontsize=14)
ax72.imshow(sinoaligsort.T[50:-50], cmap='bone', vmin=-4, vmax=1, aspect='auto')
ax72.set_xlabel('Projection number'); ax72.set_ylabel('Radial coordinate')
ax72.set_title('After horizontal alignment', fontsize=14)
plt.show()

In [ ]:
originalproj = compute_aligned_horizontal(
        valignproj, shiftproj, shift_method=params["shiftmeth"]
    )

#### Saving progress

We save the aligned projections to `PXCTalignedprojections.npz`.

Note: we save `alignedproj` — the **derivative** projections with all alignment shifts applied. Part 3 will use these for the final tomographic reconstruction.

In [ ]:
# calculate one slice for display
aligned_sinogram = np.transpose(alignedproj[:, 300, :])
oneslicefordisplay(aligned_sinogram, theta, **params)

#### Let us save our progress so far and make a break for discution/questions

In [ ]:
outputfname = "PXCTalignedprojections.npz"
np.savez(outputfname, wavelen = wavelen, psize = pixsize, projections = alignedproj, theta = theta)

In [ ]:
!ls -lrth